# Quantization and Distillation — Theory Notebook

[← Mixture of Experts and Routing](../10-Mixture-of-Experts-and-Routing/notes.md) | [Home](../../README.md) | [RAG Math and Retrieval →](../12-RAG-Math-and-Retrieval/notes.md)

Interactive implementations of quantization and knowledge distillation mathematics.

**Sections:**
1. Uniform Quantization Fundamentals
2. Symmetric vs Asymmetric Quantization
3. Quantization Error vs Bit Width
4. GPTQ-Style Weight Compensation
5. AWQ Salient Weight Scaling
6. SmoothQuant Difficulty Migration
7. Temperature Softening and Soft Labels
8. KL Divergence Distillation Loss
9. Feature and Attention Distillation
10. Quality-Efficiency Pareto Frontier

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_width=14):
    """Print a formatted ASCII table."""
    hdr = ' | '.join(h.center(col_width) for h in headers)
    print(hdr)
    print('-' * len(hdr))
    for row in rows:
        print(' | '.join(str(v).center(col_width) for v in row))

def fmt_sci(x, p=4):
    return f'{x:.{p}e}'

def fmt_vec(v, p=4):
    return '[' + ', '.join(f'{x:.{p}f}' for x in v) + ']'

def fmt_bytes(n_bytes):
    if n_bytes >= 1e9:
        return f'{n_bytes/1e9:.2f} GB'
    elif n_bytes >= 1e6:
        return f'{n_bytes/1e6:.2f} MB'
    elif n_bytes >= 1e3:
        return f'{n_bytes/1e3:.2f} KB'
    return f'{n_bytes} B'

def softmax(x, temperature=1.0):
    """Numerically stable softmax with temperature."""
    x = np.asarray(x, dtype=np.float64)
    z = x / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def kl_divergence(p, q):
    """KL(P || Q) with numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    mask = p > 1e-15
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def entropy(p):
    """Shannon entropy in nats."""
    p = np.asarray(p, dtype=np.float64)
    mask = p > 1e-15
    return -np.sum(p[mask] * np.log(p[mask]))

print('Setup complete ✓')

## §1 Uniform Quantization Fundamentals

Map floating-point weights to integers with scale $s$ and zero point $z$:

$$x_q = \text{clamp}\!\left(\text{round}\!\left(\frac{x}{s} + z\right),\; q_{\min},\; q_{\max}\right)$$

$$\hat{x} = s \cdot (x_q - z)$$

We implement quantize, dequantize, and measure error.

In [ ]:
# === §1: Uniform Quantization Fundamentals ===

def uniform_quantize(x, bits, symmetric=True):
    """Uniform quantization with scale and zero-point computation."""
    x = np.asarray(x, dtype=np.float64)
    if symmetric:
        # --- 1a: Symmetric quantization ---
        q_max = 2**(bits-1) - 1
        q_min = -2**(bits-1)
        x_absmax = np.max(np.abs(x))
        s = x_absmax / q_max if x_absmax > 0 else 1.0
        z = 0
    else:
        # --- 1b: Asymmetric quantization ---
        q_max = 2**bits - 1
        q_min = 0
        x_min, x_max = np.min(x), np.max(x)
        s = (x_max - x_min) / (q_max - q_min) if x_max > x_min else 1.0
        z = int(np.round(-x_min / s))
    
    # Quantize
    x_q = np.clip(np.round(x / s + z), q_min, q_max).astype(int)
    # Dequantize
    x_hat = s * (x_q - z)
    
    return x_q, x_hat, s, z

# --- 1c: Demo with sample weights ---
weights = np.array([-1.2, 0.3, 0.8, -0.5, 1.5, 0.1])
print('Original weights:', fmt_vec(weights))
print()

for bits in [8, 4, 3, 2]:
    x_q, x_hat, s, z = uniform_quantize(weights, bits, symmetric=True)
    mse = np.mean((weights - x_hat)**2)
    max_err = np.max(np.abs(weights - x_hat))
    print(f'--- {bits}-bit symmetric ---')
    print(f'  Scale s = {s:.6f}, Zero point z = {z}')
    print(f'  Quantized:   {list(x_q)}')
    print(f'  Dequantized: {fmt_vec(x_hat)}')
    print(f'  MSE = {fmt_sci(mse)}, Max error = {max_err:.6f}')
    print()

# --- 1d: Theoretical vs actual MSE ---
print('Theoretical MSE = s²/12 comparison:')
rows = []
for bits in [8, 4, 3, 2]:
    _, _, s, _ = uniform_quantize(weights, bits, symmetric=True)
    _, x_hat, _, _ = uniform_quantize(weights, bits, symmetric=True)
    actual_mse = np.mean((weights - x_hat)**2)
    theoretical_mse = s**2 / 12
    rows.append([f'{bits}-bit', fmt_sci(theoretical_mse), fmt_sci(actual_mse)])
print_table(['Bits', 'Theory s²/12', 'Actual MSE'], rows)

## §2 Symmetric vs Asymmetric Quantization

**Symmetric** ($z=0$): range centered at zero; best for weights (typically symmetric distributions).

**Asymmetric** ($z \neq 0$): maps $[x_{\min}, x_{\max}]$ exactly; better for non-negative activations.

We compare both on weight and activation distributions.

In [ ]:
# === §2: Symmetric vs Asymmetric Quantization ===

# --- 2a: Weight distribution (roughly symmetric, mean ≈ 0) ---
np.random.seed(42)
w = np.random.randn(10000) * 0.02  # Typical LLM weight dist

results_w = []
for bits in [8, 4, 3]:
    _, w_sym, s_sym, _ = uniform_quantize(w, bits, symmetric=True)
    _, w_asym, s_asym, z_asym = uniform_quantize(w, bits, symmetric=False)
    mse_sym = np.mean((w - w_sym)**2)
    mse_asym = np.mean((w - w_asym)**2)
    results_w.append([f'{bits}-bit', fmt_sci(mse_sym), fmt_sci(mse_asym),
                      'Sym' if mse_sym <= mse_asym else 'Asym'])

print('=== Weights (Gaussian, mean≈0) ===')
print_table(['Bits', 'Sym MSE', 'Asym MSE', 'Winner'], results_w)

# --- 2b: Activation distribution (non-negative, like post-ReLU) ---
act = np.abs(np.random.randn(10000) * 0.5) + 0.1  # Non-negative

results_a = []
for bits in [8, 4, 3]:
    _, a_sym, _, _ = uniform_quantize(act, bits, symmetric=True)
    _, a_asym, _, _ = uniform_quantize(act, bits, symmetric=False)
    mse_sym = np.mean((act - a_sym)**2)
    mse_asym = np.mean((act - a_asym)**2)
    results_a.append([f'{bits}-bit', fmt_sci(mse_sym), fmt_sci(mse_asym),
                      'Sym' if mse_sym <= mse_asym else 'Asym'])

print('\n=== Activations (non-negative) ===')
print_table(['Bits', 'Sym MSE', 'Asym MSE', 'Winner'], results_a)

print('\nKey insight: Symmetric wins for weights; Asymmetric wins for non-negative activations.')

# --- 2c: Visualise ---
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    # Weights
    bits = 4
    _, w_sym, s_s, _ = uniform_quantize(w, bits, symmetric=True)
    _, w_asym, s_a, z_a = uniform_quantize(w, bits, symmetric=False)
    axes[0].hist(w, bins=100, alpha=0.5, label='Original', density=True)
    axes[0].hist(w_sym, bins=50, alpha=0.5, label='Symmetric', density=True)
    axes[0].hist(w_asym, bins=50, alpha=0.5, label='Asymmetric', density=True)
    axes[0].set_title(f'Weights (4-bit)')
    axes[0].legend()
    # Activations
    _, a_sym, _, _ = uniform_quantize(act, bits, symmetric=True)
    _, a_asym, _, _ = uniform_quantize(act, bits, symmetric=False)
    axes[1].hist(act, bins=100, alpha=0.5, label='Original', density=True)
    axes[1].hist(a_sym, bins=50, alpha=0.5, label='Symmetric', density=True)
    axes[1].hist(a_asym, bins=50, alpha=0.5, label='Asymmetric', density=True)
    axes[1].set_title(f'Activations (4-bit)')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig('sym_vs_asym.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved sym_vs_asym.png')

## §3 Quantization Error vs Bit Width

Error analysis across bit widths. For uniform quantization:

$$\text{MSE}_{\text{round}} = \frac{s^2}{12} = \frac{R^2}{3 \cdot 2^{2b}}$$

Each additional bit reduces MSE by $4\times$. We verify this empirically on large weight arrays.

In [ ]:
# === §3: Quantization Error vs Bit Width ===

np.random.seed(42)
w_large = np.random.randn(100000) * 0.02  # 100K weights

# --- 3a: MSE vs bits ---
bit_widths = list(range(1, 17))
mses = []
theoretical_mses = []

for b in bit_widths:
    _, w_hat, s, _ = uniform_quantize(w_large, b, symmetric=True)
    mse = np.mean((w_large - w_hat)**2)
    mses.append(mse)
    theoretical_mses.append(s**2 / 12)

print('=== MSE vs Bit Width ===')
rows = []
for i, b in enumerate(bit_widths[:10]):
    ratio = mses[i-1] / mses[i] if i > 0 else float('nan')
    rows.append([b, fmt_sci(mses[i]), fmt_sci(theoretical_mses[i]),
                 f'{ratio:.2f}x' if i > 0 else '-'])
print_table(['Bits', 'Actual MSE', 'Theory MSE', 'Reduction'], rows)

print('\nExpected reduction per bit: 4.0x (theory: MSE ∝ 2^{-2b})')

# --- 3b: Memory savings ---
n_params = 8e9  # 8B parameter model
print(f'\n=== Memory for {n_params/1e9:.0f}B parameter model ===')
for b in [32, 16, 8, 4, 3, 2, 1]:
    mem = n_params * b / 8
    print(f'  {b:2d}-bit: {fmt_bytes(mem)}')

# --- 3c: Visualise ---
if HAS_MPL:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.semilogy(bit_widths, mses, 'bo-', label='Actual MSE', markersize=6)
    ax.semilogy(bit_widths, theoretical_mses, 'r--', label='Theoretical s²/12', alpha=0.7)
    ax.set_xlabel('Bit Width')
    ax.set_ylabel('MSE (log scale)')
    ax.set_title('Quantization Error vs Bit Width')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(bit_widths)
    plt.tight_layout()
    plt.savefig('mse_vs_bits.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved mse_vs_bits.png')

## §4 GPTQ-Style Weight Compensation

After quantizing weight $w_j$, update remaining weights to compensate:

$$\delta w_{j+1:} = -\frac{(w_{q,j} - w_j)}{H^{-1}_{jj}} \cdot H^{-1}_{j,j+1:}$$

This uses second-order (Hessian) information to minimise layer output reconstruction error.

In [ ]:
# === §4: GPTQ-Style Weight Compensation ===

def quantize_scalar(w, scale, bits=4, symmetric=True):
    """Quantize a single scalar weight."""
    q_max = 2**(bits-1) - 1
    q_min = -2**(bits-1)
    w_q = np.clip(np.round(w / scale), q_min, q_max) * scale
    return w_q

def gptq_quantize_row(w, H, scale, bits=4):
    """GPTQ-style quantization with Hessian compensation."""
    n = len(w)
    w_q = w.copy()
    H_inv = np.linalg.inv(H)
    errors = []
    
    for j in range(n):
        # Quantize weight j
        w_orig = w_q[j]
        w_quant = quantize_scalar(w_orig, scale, bits)
        delta = w_quant - w_orig
        errors.append(delta)
        w_q[j] = w_quant
        
        # Compensate remaining weights
        if j < n - 1:
            update = -delta / H_inv[j, j] * H_inv[j, j+1:]
            w_q[j+1:] += update
    
    return w_q, errors

# --- 4a: Small example ---
w = np.array([1.2, -0.8, 0.4, 0.7, -0.3])
# Hessian from H = 2*X*X^T (simulated)
np.random.seed(42)
X = np.random.randn(5, 100)
H = 2 * X @ X.T / 100 + 0.01 * np.eye(5)  # Damped Hessian

scale = 0.2  # INT4 scale

# Naive quantization (no compensation)
w_naive = np.array([quantize_scalar(wi, scale) for wi in w])

# GPTQ quantization (with compensation)
w_gptq, deltas = gptq_quantize_row(w.copy(), H, scale)

print('=== GPTQ vs Naive Quantization ===')
print(f'Original weights:    {fmt_vec(w)}')
print(f'Naive quantized:     {fmt_vec(w_naive)}')
print(f'GPTQ quantized:      {fmt_vec(w_gptq)}')
print()

# --- 4b: Reconstruction error comparison ---
# Output error: ||Wx - W_q x||² for random inputs
np.random.seed(0)
test_X = np.random.randn(5, 1000)  # 1000 test inputs

out_orig = w @ test_X
out_naive = w_naive @ test_X
out_gptq = w_gptq @ test_X

mse_naive = np.mean((out_orig - out_naive)**2)
mse_gptq = np.mean((out_orig - out_gptq)**2)

print(f'Output reconstruction MSE:')
print(f'  Naive:  {fmt_sci(mse_naive)}')
print(f'  GPTQ:   {fmt_sci(mse_gptq)}')
print(f'  Ratio:  {mse_naive/mse_gptq:.2f}x improvement')

# --- 4c: Larger example ---
np.random.seed(42)
d = 64
w_big = np.random.randn(d) * 0.5
X_big = np.random.randn(d, 500)
H_big = 2 * X_big @ X_big.T / 500 + 0.01 * np.eye(d)

w_naive_big = np.array([quantize_scalar(wi, 0.1) for wi in w_big])
w_gptq_big, _ = gptq_quantize_row(w_big.copy(), H_big, 0.1)

test_big = np.random.randn(d, 2000)
mse_n = np.mean((w_big @ test_big - w_naive_big @ test_big)**2)
mse_g = np.mean((w_big @ test_big - w_gptq_big @ test_big)**2)

print(f'\n=== Larger Example (d={d}) ===')
print(f'  Naive MSE:  {fmt_sci(mse_n)}')
print(f'  GPTQ MSE:   {fmt_sci(mse_g)}')
print(f'  Improvement: {mse_n/mse_g:.2f}x')

## §5 AWQ Salient Weight Scaling

AWQ scales weights by activation magnitude before quantization:

$$\hat{W}_i = W_i \cdot s_i, \quad \hat{X}_i = X_i / s_i$$

where $s_i = \text{mean}(|X_i|)^\alpha$. Salient channels (high activation) get finer quantization.

In [ ]:
# === §5: AWQ Salient Weight Scaling ===

def awq_quantize(W, X, bits=4, alpha=0.5):
    """AWQ-style activation-aware weight quantization."""
    # --- 5a: Compute per-channel activation scales ---
    act_scales = np.mean(np.abs(X), axis=1)  # Per input channel
    
    # Search for optimal alpha
    best_mse = float('inf')
    best_alpha = alpha
    
    for a in np.linspace(0.0, 1.0, 21):
        s = act_scales ** a
        s = np.maximum(s, 1e-8)  # Avoid zero
        
        # Scale weights up, inputs down
        W_scaled = W * s[np.newaxis, :]  # (m, n) * (n,)
        X_scaled = X / s[:, np.newaxis]   # (n, T) / (n, 1)
        
        # Quantize scaled weights
        W_q = np.zeros_like(W_scaled)
        for row in range(W_scaled.shape[0]):
            _, w_hat, _, _ = uniform_quantize(W_scaled[row], bits, symmetric=True)
            W_q[row] = w_hat
        
        # Descale and compute output error
        W_q_descaled = W_q / s[np.newaxis, :]
        out_orig = W @ X
        out_q = W_q_descaled @ X
        mse = np.mean((out_orig - out_q)**2)
        
        if mse < best_mse:
            best_mse = mse
            best_alpha = a
    
    # Apply best alpha
    s = act_scales ** best_alpha
    s = np.maximum(s, 1e-8)
    W_scaled = W * s[np.newaxis, :]
    W_q_final = np.zeros_like(W_scaled)
    for row in range(W_scaled.shape[0]):
        _, w_hat, _, _ = uniform_quantize(W_scaled[row], bits, symmetric=True)
        W_q_final[row] = w_hat
    W_q_descaled = W_q_final / s[np.newaxis, :]
    
    return W_q_descaled, best_alpha, best_mse

# --- 5b: Demo ---
np.random.seed(42)
m, n, T = 32, 64, 200
W = np.random.randn(m, n) * 0.02
X = np.random.randn(n, T) * 0.5

# Add outlier channels (simulate transformer activations)
outlier_channels = [5, 20, 45]
for ch in outlier_channels:
    X[ch, :] *= 10  # 10x larger activations

# Naive quantization
W_naive_q = np.zeros_like(W)
for row in range(m):
    _, w_hat, _, _ = uniform_quantize(W[row], 4, symmetric=True)
    W_naive_q[row] = w_hat

out_orig = W @ X
mse_naive = np.mean((out_orig - W_naive_q @ X)**2)

# AWQ quantization
W_awq, best_alpha, mse_awq = awq_quantize(W, X, bits=4)
mse_awq_actual = np.mean((out_orig - W_awq @ X)**2)

print('=== AWQ vs Naive (4-bit, with outlier channels) ===')
print(f'Outlier channels: {outlier_channels}')
print(f'Naive MSE:  {fmt_sci(mse_naive)}')
print(f'AWQ MSE:    {fmt_sci(mse_awq_actual)} (alpha={best_alpha:.2f})')
print(f'Improvement: {mse_naive/mse_awq_actual:.2f}x')
print()

# --- 5c: Alpha search landscape ---
alphas = np.linspace(0, 1, 21)
mses_alpha = []
for a in alphas:
    act_s = np.mean(np.abs(X), axis=1) ** a
    act_s = np.maximum(act_s, 1e-8)
    Ws = W * act_s[np.newaxis, :]
    Wq = np.zeros_like(Ws)
    for r in range(m):
        _, wh, _, _ = uniform_quantize(Ws[r], 4, symmetric=True)
        Wq[r] = wh
    Wq_d = Wq / act_s[np.newaxis, :]
    mses_alpha.append(np.mean((out_orig - Wq_d @ X)**2))

print('Alpha search:')
for i in range(0, len(alphas), 4):
    print(f'  alpha={alphas[i]:.2f}  MSE={fmt_sci(mses_alpha[i])}')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(alphas, mses_alpha, 'b.-')
    ax.axvline(best_alpha, color='r', linestyle='--', label=f'Best α={best_alpha:.2f}')
    ax.set_xlabel('Alpha (scaling exponent)')
    ax.set_ylabel('Output MSE')
    ax.set_title('AWQ: Output Error vs Scaling Exponent')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('awq_alpha.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved awq_alpha.png')

## §6 SmoothQuant Difficulty Migration

Migrate quantization difficulty from activations to weights:

$$Y = X W = (X \cdot \text{diag}(s)^{-1})(\text{diag}(s) \cdot W) = \hat{X}\hat{W}$$

$$s_j = \frac{\max(|X_j|)^\alpha}{\max(|W_j|)^{1-\alpha}}$$

After smoothing, both $\hat{X}$ and $\hat{W}$ are easier to quantize (W8A8).

In [ ]:
# === §6: SmoothQuant Difficulty Migration ===

def smoothquant(X, W, bits=8, alpha=0.5):
    """SmoothQuant: migrate quantization difficulty from acts to weights."""
    n = X.shape[0]  # = W.shape[1] (input dim)
    
    # --- 6a: Compute per-channel scales ---
    act_max = np.max(np.abs(X), axis=1)   # Per input channel
    w_max = np.max(np.abs(W), axis=0)     # Per input channel (column)
    
    s = (act_max ** alpha) / (np.maximum(w_max, 1e-8) ** (1 - alpha))
    s = np.maximum(s, 1e-8)
    
    # Apply smoothing
    X_smooth = X / s[:, np.newaxis]  # Divide activations
    W_smooth = W * s[np.newaxis, :]  # Multiply weights (note: W is m×n)
    
    return X_smooth, W_smooth, s

# --- 6b: Setup with outlier activations ---
np.random.seed(42)
n_in, n_out, T = 64, 32, 500
X = np.random.randn(n_in, T) * 0.5   # (n_in, T)
W = np.random.randn(n_out, n_in) * 0.02  # (n_out, n_in)

# Add activation outliers in specific channels
outlier_chs = [5, 15, 30, 50]
for ch in outlier_chs:
    X[ch, :] *= 20  # 20x larger

Y_orig = W @ X  # True output

# --- 6c: Quantize without smoothing (W8A8) ---
W_q = np.zeros_like(W)
for r in range(n_out):
    _, w_hat, _, _ = uniform_quantize(W[r], 8, symmetric=True)
    W_q[r] = w_hat

X_q = np.zeros_like(X)
for ch in range(n_in):
    _, x_hat, _, _ = uniform_quantize(X[ch], 8, symmetric=True)
    X_q[ch] = x_hat

mse_no_smooth = np.mean((Y_orig - W_q @ X_q)**2)

# --- 6d: Quantize with SmoothQuant ---
best_mse = float('inf')
best_a = 0.5
for a in np.linspace(0.1, 0.9, 9):
    X_s, W_s, s = smoothquant(X, W, alpha=a)
    
    W_sq = np.zeros_like(W_s)
    for r in range(n_out):
        _, w_hat, _, _ = uniform_quantize(W_s[r], 8, symmetric=True)
        W_sq[r] = w_hat
    
    X_sq = np.zeros_like(X_s)
    for ch in range(n_in):
        _, x_hat, _, _ = uniform_quantize(X_s[ch], 8, symmetric=True)
        X_sq[ch] = x_hat
    
    mse = np.mean((Y_orig - W_sq @ X_sq)**2)
    if mse < best_mse:
        best_mse = mse
        best_a = a

print('=== SmoothQuant W8A8 (with outlier activations) ===')
print(f'Outlier channels: {outlier_chs}')
print(f'Without SmoothQuant: MSE = {fmt_sci(mse_no_smooth)}')
print(f'With SmoothQuant:    MSE = {fmt_sci(best_mse)} (alpha={best_a:.1f})')
print(f'Improvement: {mse_no_smooth/best_mse:.2f}x')
print()

# --- 6e: Show dynamic range before/after smoothing ---
X_s, W_s, s = smoothquant(X, W, alpha=best_a)
print('Dynamic range (max/min magnitude) per-channel:')
x_range = np.max(np.abs(X), axis=1) / (np.min(np.abs(X), axis=1) + 1e-10)
xs_range = np.max(np.abs(X_s), axis=1) / (np.min(np.abs(X_s), axis=1) + 1e-10)
print(f'  Activations before: max ratio = {np.max(x_range):.1f}')
print(f'  Activations after:  max ratio = {np.max(xs_range):.1f}')
w_range = np.max(np.abs(W)) / (np.min(np.abs(W[W != 0])) + 1e-10)
ws_range = np.max(np.abs(W_s)) / (np.min(np.abs(W_s[W_s != 0])) + 1e-10)
print(f'  Weights before:     range = {w_range:.1f}')
print(f'  Weights after:      range = {ws_range:.1f}')

## §7 Temperature Softening and Soft Labels

Softmax at temperature $\tau$:

$$p_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

Higher $\tau$ reveals "dark knowledge" — the relative probabilities of non-top classes
that encode similarity structure.

In [ ]:
# === §7: Temperature Softening and Soft Labels ===

logits = np.array([4.0, 2.0, 0.5, -1.0, -2.5])
labels = ['Paris', 'Lyon', 'Rome', 'Berlin', 'Tokyo']

# --- 7a: Temperature sweep ---
temperatures = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 20.0]

print('=== Temperature Effect on Softmax ===')
print(f'Logits: {list(logits)}')
print(f'Labels: {labels}\n')

rows = []
for tau in temperatures:
    p = softmax(logits, tau)
    H = entropy(p)
    max_entropy = np.log(len(logits))
    rows.append([f'{tau:.1f}',
                 f'{p[0]:.4f}',
                 f'{p[1]:.4f}',
                 f'{p[2]:.4f}',
                 f'{H:.4f}',
                 f'{H/max_entropy*100:.1f}%'])
print_table(['τ', 'P(Paris)', 'P(Lyon)', 'P(Rome)', 'Entropy', '% Max H'], rows)

# --- 7b: Dark knowledge ---
print('\n=== Dark Knowledge Analysis ===')
p1 = softmax(logits, 1.0)
p5 = softmax(logits, 5.0)
print(f'At τ=1: P(Lyon)/P(Rome) = {p1[1]/p1[2]:.2f} (Lyon {p1[1]/p1[2]:.1f}x more likely)')
print(f'At τ=5: P(Lyon)/P(Rome) = {p5[1]/p5[2]:.2f} (Lyon {p5[1]/p5[2]:.1f}x more likely)')
print(f'Dark knowledge: Lyon is geographically/linguistically closer to Paris than Rome.')
print(f'This relationship is preserved at all temperatures but MORE VISIBLE at high τ.')

# --- 7c: Information content quantification ---
print('\n=== Information in Soft Labels ===')
for tau in [1.0, 2.0, 5.0, 10.0]:
    p = softmax(logits, tau)
    # Non-zero probabilities carry information
    effective_classes = np.exp(entropy(p))
    print(f'τ={tau:5.1f}: effective classes = {effective_classes:.2f} / {len(logits)}')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Distribution at different temperatures
    for tau in [0.5, 1.0, 2.0, 5.0, 10.0]:
        p = softmax(logits, tau)
        axes[0].plot(range(len(labels)), p, 'o-', label=f'τ={tau}', markersize=6)
    axes[0].set_xticks(range(len(labels)))
    axes[0].set_xticklabels(labels, rotation=30)
    axes[0].set_ylabel('Probability')
    axes[0].set_title('Softmax Distribution at Different Temperatures')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Entropy vs temperature
    taus = np.linspace(0.1, 20, 100)
    entropies = [entropy(softmax(logits, t)) for t in taus]
    axes[1].plot(taus, entropies, 'b-')
    axes[1].axhline(np.log(len(logits)), color='r', linestyle='--',
                    label=f'Max entropy (uniform) = {np.log(len(logits)):.2f}')
    axes[1].set_xlabel('Temperature τ')
    axes[1].set_ylabel('Entropy (nats)')
    axes[1].set_title('Entropy vs Temperature')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('temperature_softening.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved temperature_softening.png')

## §8 KL Divergence Distillation Loss

Hinton distillation loss:

$$\mathcal{L} = \alpha\, \mathcal{L}_{\text{CE}}(y, y_{\text{true}}) + (1-\alpha)\, \tau^2\, D_{KL}(\sigma(z_T/\tau) \| \sigma(z_S/\tau))$$

The $\tau^2$ factor compensates for gradient scaling at high temperature.

In [ ]:
# === §8: KL Divergence Distillation Loss ===

def distillation_loss(z_teacher, z_student, true_label, tau=2.0, alpha=0.5):
    """Compute Hinton distillation loss."""
    # Soft distributions at temperature tau
    p_T = softmax(z_teacher, tau)
    p_S = softmax(z_student, tau)
    
    # KL divergence (teacher || student)
    kl = kl_divergence(p_T, p_S)
    
    # Hard cross-entropy loss
    p_S_hard = softmax(z_student, 1.0)
    ce_loss = -np.log(p_S_hard[true_label] + 1e-15)
    
    # Combined loss
    total = alpha * ce_loss + (1 - alpha) * tau**2 * kl
    
    return {
        'p_T': p_T, 'p_S': p_S,
        'kl_div': kl, 'ce_loss': ce_loss,
        'distil_loss': tau**2 * kl,
        'total_loss': total
    }

# --- 8a: Teacher and student logits ---
z_T = np.array([3.0, 1.0, -0.5, 0.2])
z_S = np.array([2.5, 0.8, 0.0, 0.1])
true_label = 0
class_names = ['cat', 'dog', 'bird', 'fish']

print('=== Distillation Loss Computation ===')
print(f'Teacher logits: {list(z_T)}')
print(f'Student logits: {list(z_S)}')
print(f'True label: {class_names[true_label]}\n')

# Different temperatures
rows = []
for tau in [1.0, 2.0, 3.0, 5.0, 10.0]:
    r = distillation_loss(z_T, z_S, true_label, tau=tau, alpha=0.5)
    rows.append([f'{tau:.0f}', f'{r["kl_div"]:.6f}', f'{r["distil_loss"]:.6f}',
                 f'{r["ce_loss"]:.4f}', f'{r["total_loss"]:.6f}'])

print_table(['τ', 'KL(T||S)', 'τ²·KL', 'CE Loss', 'Total'], rows)

# --- 8b: Detailed breakdown at τ=2 ---
tau = 2.0
r = distillation_loss(z_T, z_S, true_label, tau=tau, alpha=0.5)
print(f'\n=== Detailed Breakdown at τ={tau} ===')
print(f'Teacher soft probs: {fmt_vec(r["p_T"])}')
print(f'Student soft probs: {fmt_vec(r["p_S"])}')
print(f'KL(T || S) = {r["kl_div"]:.6f}')
print(f'τ² · KL    = {tau**2:.0f} × {r["kl_div"]:.6f} = {r["distil_loss"]:.6f}')
print(f'CE(hard)   = {r["ce_loss"]:.6f}')
print(f'Total      = 0.5 × {r["ce_loss"]:.6f} + 0.5 × {r["distil_loss"]:.6f} = {r["total_loss"]:.6f}')

# --- 8c: Forward KL vs Reverse KL ---
print('\n=== Forward vs Reverse KL at τ=3 ===')
tau = 3.0
p_T = softmax(z_T, tau)
p_S = softmax(z_S, tau)
fwd_kl = kl_divergence(p_T, p_S)  # Standard: D_KL(T || S)
rev_kl = kl_divergence(p_S, p_T)  # Reverse:  D_KL(S || T)
js = 0.5 * kl_divergence(p_T, 0.5*(p_T+p_S)) + 0.5 * kl_divergence(p_S, 0.5*(p_T+p_S))
print(f'Forward KL D(T||S) = {fwd_kl:.6f}  (mode-covering)')
print(f'Reverse KL D(S||T) = {rev_kl:.6f}  (mode-seeking)')
print(f'Jensen-Shannon     = {js:.6f}  (symmetric, bounded)')

## §9 Feature and Attention Distillation

Beyond logit matching, distil intermediate representations:

$$\mathcal{L}_{\text{feat}} = \sum_l \|h_S^{(l)} - W_l h_T^{(l')}\|_F^2$$

$$\mathcal{L}_{\text{attn}} = \sum_l \|A_T^{(l)} - A_S^{(l)}\|_F^2$$

We simulate a multi-layer teacher-student and measure feature alignment.

In [ ]:
# === §9: Feature and Attention Distillation ===

np.random.seed(42)

# --- 9a: Simulate teacher and student hidden states ---
d_teacher = 768   # Teacher hidden dim
d_student = 256   # Student hidden dim
seq_len = 20      # Sequence length
n_layers_teacher = 12
n_layers_student = 4

# Teacher hidden states (one per layer)
teacher_hiddens = [np.random.randn(seq_len, d_teacher) * 0.5
                   for _ in range(n_layers_teacher)]

# Student hidden states
student_hiddens = [np.random.randn(seq_len, d_student) * 0.5
                   for _ in range(n_layers_student)]

# --- 9b: Layer mapping strategy (uniform spacing) ---
# Map student layer i → teacher layer (i+1) * L_T / L_S - 1
layer_map = {}
for s_layer in range(n_layers_student):
    t_layer = (s_layer + 1) * n_layers_teacher // n_layers_student - 1
    layer_map[s_layer] = t_layer

print('=== Layer Mapping (Uniform Spacing) ===')
for s, t in layer_map.items():
    print(f'  Student layer {s} ← Teacher layer {t}')

# --- 9c: Feature distillation loss ---
def feature_distillation_loss(student_h, teacher_h, d_s, d_t):
    """Compute feature distillation loss with learned projection."""
    # Random projection W_proj: d_t → d_s (simulating learned projection)
    np.random.seed(123)
    W_proj = np.random.randn(d_t, d_s) / np.sqrt(d_t)
    
    # Project teacher to student dimension
    teacher_proj = teacher_h @ W_proj  # (seq, d_s)
    
    # MSE loss
    loss = np.mean((student_h - teacher_proj)**2)
    return loss

total_feat_loss = 0
print('\n=== Feature Distillation Losses ===')
for s_layer, t_layer in layer_map.items():
    loss = feature_distillation_loss(
        student_hiddens[s_layer], teacher_hiddens[t_layer],
        d_student, d_teacher
    )
    total_feat_loss += loss
    print(f'  Layer {s_layer}→{t_layer}: MSE = {loss:.6f}')
print(f'  Total feature loss: {total_feat_loss:.6f}')

# --- 9d: Attention transfer ---
n_heads_teacher = 12
n_heads_student = 4

# Simulate attention matrices
def make_attention(seq_len, n_heads):
    """Create random attention matrices (row-stochastic)."""
    logits = np.random.randn(n_heads, seq_len, seq_len)
    # Apply causal mask
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * (-1e9)
    logits += mask[np.newaxis, :, :]
    # Softmax per row
    att = np.exp(logits - logits.max(axis=-1, keepdims=True))
    att = att / att.sum(axis=-1, keepdims=True)
    return att

teacher_attn = [make_attention(seq_len, n_heads_teacher)
                for _ in range(n_layers_teacher)]
student_attn = [make_attention(seq_len, n_heads_student)
                for _ in range(n_layers_student)]

print('\n=== Attention Distillation Losses ===')
total_attn_loss = 0
for s_layer, t_layer in layer_map.items():
    # Average teacher heads to match student head count
    heads_per_group = n_heads_teacher // n_heads_student
    t_attn_avg = teacher_attn[t_layer].reshape(
        n_heads_student, heads_per_group, seq_len, seq_len
    ).mean(axis=1)
    
    loss = np.mean((student_attn[s_layer] - t_attn_avg)**2)
    total_attn_loss += loss
    print(f'  Layer {s_layer}→{t_layer}: Attn MSE = {loss:.6f}')
print(f'  Total attention loss: {total_attn_loss:.6f}')

# --- 9e: Combined distillation loss ---
print('\n=== Combined Multi-Level Distillation ===')
beta_logit = 1.0
beta_feat = 0.1
beta_attn = 0.01
logit_loss = 0.025  # Simulated logit distillation loss
combined = beta_logit * logit_loss + beta_feat * total_feat_loss + beta_attn * total_attn_loss
print(f'  Logit loss:    {beta_logit} × {logit_loss:.4f}     = {beta_logit*logit_loss:.6f}')
print(f'  Feature loss:  {beta_feat} × {total_feat_loss:.4f} = {beta_feat*total_feat_loss:.6f}')
print(f'  Attention loss: {beta_attn} × {total_attn_loss:.4f} = {beta_attn*total_attn_loss:.6f}')
print(f'  Combined:      {combined:.6f}')

## §10 Quality-Efficiency Pareto Frontier

Map different quantization and distillation configurations onto a quality-cost Pareto frontier.
Pareto-optimal points: no other configuration achieves better quality at same cost (or lower cost at same quality).

In [ ]:
# === §10: Quality-Efficiency Pareto Frontier ===

# --- 10a: Define model configurations ---
# (name, params_B, bits, quality_score_0_100, memory_GB, tokens_per_sec)
configs = [
    ('70B BF16',       70,  16, 95.0, 140.0,  15),
    ('70B INT8',       70,   8, 94.0,  70.0,  30),
    ('70B INT4 GPTQ',  70,   4, 91.0,  35.0,  55),
    ('70B INT4 AWQ',   70,   4, 91.5,  35.0,  60),
    ('70B 2-bit',      70,   2, 78.0,  17.5,  90),
    ('13B BF16',       13,  16, 85.0,  26.0,  60),
    ('13B INT4',       13,   4, 83.0,   6.5, 120),
    ('7B BF16',         7,  16, 80.0,  14.0, 100),
    ('7B INT8',         7,   8, 79.5,   7.0, 180),
    ('7B INT4',         7,   4, 77.0,   3.5, 250),
    ('7B distil BF16',  7,  16, 84.0,  14.0, 100),
    ('7B distil INT4',  7,   4, 81.0,   3.5, 250),
    ('3B BF16',         3,  16, 72.0,   6.0, 200),
    ('3B distil INT4',  3,   4, 76.0,   1.5, 500),
    ('1B distil INT4',  1,   4, 65.0,   0.5, 800),
]

print('=== Model Configuration Space ===')
rows = []
for name, params, bits, quality, mem, tps in configs:
    rows.append([name, f'{params}B', f'{bits}b', f'{quality:.1f}',
                 f'{mem:.1f} GB', f'{tps}'])
print_table(['Config', 'Params', 'Bits', 'Quality', 'Memory', 'Tok/s'], rows, col_width=16)

# --- 10b: Find Pareto frontier (quality vs memory) ---
def find_pareto(points):
    """Find Pareto-optimal points (minimise cost, maximise quality)."""
    # points: list of (cost, quality)
    sorted_pts = sorted(enumerate(points), key=lambda x: x[1][0])
    pareto = []
    best_quality = -float('inf')
    for idx, (cost, quality) in sorted_pts:
        if quality > best_quality:
            pareto.append(idx)
            best_quality = quality
    return pareto

# Quality vs Memory Pareto
points_mem = [(mem, quality) for _, _, _, quality, mem, _ in configs]
pareto_mem = find_pareto(points_mem)

print('\n=== Pareto Frontier (Quality vs Memory) ===')
for i in pareto_mem:
    name = configs[i][0]
    quality = configs[i][3]
    mem = configs[i][4]
    print(f'  ★ {name:20s}  Quality={quality:.1f}  Memory={mem:.1f} GB')

# --- 10c: Key insight: distillation shifts frontier ---
print('\n=== Distillation Shifts the Pareto Frontier ===')
# Compare: 7B vanilla vs 7B distilled at same memory
print('At ~3.5 GB (INT4):')
print(f'  7B vanilla INT4:  quality = 77.0')
print(f'  7B distilled INT4: quality = 81.0  (+4.0 from distillation)')
print('\nDistillation provides "free" quality at same cost.')

# --- 10d: Cost-per-quality analysis ---
print('\n=== Cost Per Quality Point ===')
rows = []
for name, params, bits, quality, mem, tps in configs:
    cost_per_q = mem / quality  # GB per quality point
    rows.append([name, f'{quality:.1f}', f'{mem:.1f} GB',
                 f'{cost_per_q:.3f}'])
rows.sort(key=lambda r: float(r[3]))
print_table(['Config', 'Quality', 'Memory', 'GB/QPoint'], rows[:8], col_width=18)
print('\nMost efficient: distilled + quantized models')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (name, params, bits, quality, mem, tps) in enumerate(configs):
        marker = '★' if i in pareto_mem else 'o'
        color = 'red' if 'distil' in name else ('blue' if bits <= 4 else 'green')
        ax.scatter(mem, quality, c=color, s=100, zorder=5)
        ax.annotate(name, (mem, quality), textcoords='offset points',
                    xytext=(5, 5), fontsize=7)
    
    # Draw Pareto frontier
    pareto_points = sorted([(configs[i][4], configs[i][3]) for i in pareto_mem])
    ax.plot([p[0] for p in pareto_points], [p[1] for p in pareto_points],
            'k--', alpha=0.5, label='Pareto frontier')
    
    ax.set_xlabel('Memory (GB)')
    ax.set_ylabel('Quality Score')
    ax.set_title('Quality-Memory Pareto Frontier')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pareto_frontier.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved pareto_frontier.png')

print()
print('=' * 65)
print('   THEORY NOTEBOOK COMPLETE')
print('=' * 65)
print()
print('Key Takeaways:')
print('  1. Uniform quantization: MSE = s²/12; halves with each extra bit')
print('  2. Symmetric for weights, asymmetric for activations')
print('  3. GPTQ: Hessian compensation reduces output error significantly')
print('  4. AWQ: activation-aware scaling protects salient channels')
print('  5. SmoothQuant: migrates difficulty for W8A8 quantization')
print('  6. Temperature τ reveals dark knowledge in soft labels')
print('  7. τ² scaling compensates for gradient magnitude at high T')
print('  8. Feature + attention distillation adds complementary signal')
print('  9. Distillation shifts the Pareto frontier: same cost, higher quality')
print(' 10. Best deployment: distill THEN quantize for maximum efficiency')